In [8]:
import sys
!{sys.executable} -m ensurepip --upgrade

Looking in links: /var/folders/dd/tpdzgsps29l6xqx8hm7z2_rr0000gp/T/tmpi9m8x7fp


In [9]:
# Install dependencies
%pip install anthropic python-dotenv


[notice] A new release of pip is available: 25.1.1 -> 26.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
# Load env vars

from dotenv import load_dotenv

load_dotenv()

True

In [11]:
# Create an API client

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [12]:
from anthropic.types import MessageParam
from collections.abc import Iterable

def add_user_message(messages: list[MessageParam], text: str):
    user_message: MessageParam = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages: list[MessageParam], text: str):
    assistant_message: MessageParam = {"role": "assistant", "content": text }
    messages.append(assistant_message)

from anthropic.types import TextBlock, Message

def get_message_text(message: Message):
    return next(
        (block.text for block in message.content if isinstance(block, TextBlock)),
        "" # empty string if none
    )

def chat(messages: Iterable[MessageParam], system: str|None = None, temperature = 1.0, stop_sequences: list[str] = []):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return get_message_text(message)

In [13]:
import json
from typing import TypedDict

class TaskObj(TypedDict):
    task: str

with open('data/007_generate_eval_dataset-tasks.json', 'r', encoding='utf-8') as file:
    dataset: list[TaskObj] = json.load(file)

print(dataset)

[{'task': "Write a Python function that extracts the AWS region from an S3 bucket URI (e.g., 's3://my-bucket-us-east-1/path'). The function should return the region code or None if not found."}, {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific S3 bucket named 'my-data-bucket'. Include the necessary Version, Statement, Effect, Action, and Resource fields."}, {'task': "Write a regular expression that matches valid AWS EC2 instance IDs (format: i- followed by 17 hexadecimal characters, e.g., 'i-0abcd1234efgh5678')."}]


In [14]:
def run_prompt(test_case: TaskObj):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [15]:
results = run_eval(dataset)

In [16]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# Solution: Extract AWS Region from S3 Bucket URI\n\nHere's a comprehensive solution with multiple approaches:\n\n```python\nimport re\nfrom typing import Optional\n\ndef extract_region_from_s3_uri(s3_uri: str) -> Optional[str]:\n    \"\"\"\n    Extracts AWS region from an S3 bucket URI.\n    \n    Supports multiple S3 URI formats:\n    - s3://bucket-name/path\n    - s3://bucket-name-region/path\n    - https://bucket-name.s3.region.amazonaws.com/path\n    - https://s3.region.amazonaws.com/bucket-name/path\n    \n    Args:\n        s3_uri: S3 URI string\n        \n    Returns:\n        AWS region code (e.g., 'us-east-1') or None if not found\n    \"\"\"\n    if not s3_uri:\n        return None\n    \n    # List of valid AWS regions\n    valid_regions = {\n        'us-east-1', 'us-east-2', 'us-west-1', 'us-west-2',\n        'eu-west-1', 'eu-west-2', 'eu-west-3', 'eu-central-1',\n        'ap-northeast-1', 'ap-northeast-2', 'ap-northeast-3',\n        'ap-southeast-1', 